In [1]:
!pip install -q crewai crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48

In [3]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

from crewai import LLM
llm = LLM(
    model = "gemini/gemini-flash-lite-latest",
    temperature = 0.2
)

In [4]:
job_description = """Junior Python Developer.
Must have: Python, SQL, basic Git.
Nice to have : Django or Flask, any internship.
0-2 years experience."""

resumes = {
    "anjali.txt": """Anjali Rao. B.Tech CSE 2025.
Skills: Python, SQL, Git, Flask.
Internship: 3 months backend intern at a startup, built REST APIs in Flask.
Project: Library management system with Python and MySQL.""",

    "rohit.txt": """Rohit Verma. B.Com 2024.
Skills: MS Excel, Tally, basic accounting.
Experience: 1 year as accounts assistant.
Interested in moving into tech.""",

    "meera.txt": """Meera Iyer. MCA 2025.
Skills: Python, basic SQL.
Project: Student attendance app in Python.
No internship yet. Learning Git.""",
}

for name, text in resumes.items():
  with open(name, "w") as f:
    f.write(text)


print("Saved : ",list(resumes.keys()))

Saved :  ['anjali.txt', 'rohit.txt', 'meera.txt']


In [5]:
from crewai import Agent, Task, Crew
from crewai.flow.flow import Flow, start, listen, router, or_
import csv

class ResumeFlow(Flow):

    @start()
    def load_resume(self):
        with open(self.state["file"]) as f:
            self.state["resume"] = f.read()
        print("\n📄 Loaded:", self.state["file"])

    @listen(load_resume)
    def score_resume(self):
        answer = llm.call(
            "Score how well this resume fits the job, from 0 to 10. "
            "Reply with only the number.\n\n"
            "Job:\n" + job_description + "\n\nResume:\n" + self.state["resume"]
        )
        try:
            self.state["score"] = int(answer.strip())
        except ValueError:
            self.state["score"] = 0
        print("🔢 Score:", self.state["score"])

    @router(score_resume)
    def check_score(self):
        if self.state["score"] >= 7:
            return "shortlist"
        return "reject"

    @listen("shortlist")
    async def summarize(self):
        recruiter = Agent(
            role="Technical Recruiter",
            goal="Summarise why a candidate fits a job and what is missing",
            backstory="You are fair and specific, and only mention skills "
                      "actually written in the resume.",
            llm=llm,
            verbose=False,
        )
        task = Task(
            description="Job:\n{job}\n\nResume:\n{resume}\n\n"
                        "Write 3 strengths and 2 gaps of this candidate.",
            expected_output="3 strengths and 2 gaps.",
            agent=recruiter,
        )
        crew = Crew(agents=[recruiter], tasks=[task], verbose=False)
        result = await crew.kickoff_async(inputs={"job": job_description,
                                                  "resume": self.state["resume"]})
        self.state["decision"] = "Shortlisted"
        self.state["notes"] = result.raw

    @listen("reject")
    def draft_rejection(self):
        email = llm.call(
            "Write a short, polite rejection email (3 lines) for this candidate "
            "applying to a Junior Python Developer role.\n\nResume:\n"
            + self.state["resume"]
        )
        self.state["decision"] = "Rejected"
        self.state["notes"] = email.strip()

    @listen(or_(summarize, draft_rejection))
    def save_results(self):
        with open("results.csv", "a", newline="") as f:
            csv.writer(f).writerow([self.state["file"], self.state["score"],
                                    self.state["decision"], self.state["notes"]])
        print(self.state["decision"])

In [6]:
import os

if os.path.exists("results.csv"):
    os.remove("results.csv")

for file in resumes.keys():
    flow = ResumeFlow()
    await flow.kickoff_async(inputs={"file": file})

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ResumeFlow                                                                                               │
│  ID: 5fa7c701-e7c3-4375-801d-105a7ccdc254                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ResumeFlow                                                                                               │
│  ID: 5fa7c701-e7c3-4375-801d-105a7ccdc254                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: load_resume                                                                                            │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


📄 Loaded: anjali.txt


╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: load_resume                                                                                            │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Shortlisted


╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: summarize                                                                                              │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: save_results                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: save_results                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ResumeFlow                                                                                               │
│  ID: 5db935d5-36bb-4be9-a382-0044aae03ac4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
import pandas as pd

df = pd.read_csv("results.csv", names=["file", "score", "decision", "notes"])
df

,file,score,decision,notes
0,anjali.txt,10,Shortlisted,**Candidate Evaluation for Junior Python Devel...
1,rohit.txt,0,Rejected,"Dear Rohit,\n\nThank you for your interest in ..."
2,meera.txt,7,Shortlisted,"As a Technical Recruiter, here is my assessmen..."


In [8]:
for i, row in df.iterrows():
    print("=" * 60)
    print(row["file"], "| Score:", row["score"], "|", row["decision"])
    print("-" * 60)
    print(row["notes"])

anjali.txt | Score: 10 | Shortlisted
------------------------------------------------------------
**Candidate Evaluation for Junior Python Developer**

**Strengths:**
1. **Core Technical Stack Alignment:** Anjali possesses all the "Must have" skills required for the role, specifically Python, SQL, and Git.
2. **Framework Experience:** She meets the "Nice to have" criteria by having practical experience with Flask, which directly aligns with the job's secondary requirements.
3. **Relevant Industry Exposure:** She completed a 3-month backend internship at a startup where she built REST APIs in Flask, fulfilling the "any internship" nice-to-have criteria and demonstrating early hands-on industry experience well within the 0-2 years experience bracket.

**Gaps:**
1. **Alternative Framework Knowledge:** While she knows Flask, she does not mention experience with Django (the other option listed in the "Nice to have" criteria).
2. **Broader Database Experience:** Her project specifies MySQL, 